# ML Metadata Lab - Tracking the Iris Prediction Pipeline

**Hrishikesh Prabhu | IE-7374 MLOps | Spring 2026**

In my FastAPI lab, I built an Iris Flower Prediction API using a Random Forest model with confidence scores and species descriptions. This notebook uses ML Metadata (MLMD) to track that entire pipeline — from dataset to trained model to API deployment.

This connects directly to my previous work:
- **GitHub Lab 1**: Learned testing and CI/CD
- **FastAPI Lab**: Built the prediction API with Random Forest, confidence scores, and species info
- **This Lab**: Tracking the metadata of that same pipeline so we know exactly what data, model, and steps were involved

In [ ]:
from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2
import pandas as pd
import numpy as np
import json
import os
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("All imports successful!")

'''**Cell 3 (Markdown):**'''
## Step 1: Recreate the Dataset from FastAPI Lab
This is the same Iris dataset and split we used in the FastAPI lab.

'''**Cell 3 (Markdown):**'''
## Step 1: Recreate the Dataset from FastAPI Lab
This is the same Iris dataset and split we used in the FastAPI lab.

In [ ]:
# Load Iris dataset (same as FastAPI lab)
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
species_names = {0: "Setosa", 1: "Versicolor", 2: "Virginica"}

# Same split as FastAPI lab (70-30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Save datasets
os.makedirs('pipeline_data/train', exist_ok=True)
os.makedirs('pipeline_data/eval', exist_ok=True)

train_df = pd.DataFrame(X_train, columns=feature_names)
train_df['species'] = [species_names[label] for label in y_train]
train_df['species_id'] = y_train
train_df.to_csv('pipeline_data/train/iris_train.csv', index=False)

eval_df = pd.DataFrame(X_test, columns=feature_names)
eval_df['species'] = [species_names[label] for label in y_test]
eval_df['species_id'] = y_test
eval_df.to_csv('pipeline_data/eval/iris_eval.csv', index=False)

print(f"Training samples: {len(train_df)}")
print(f"Evaluation samples: {len(eval_df)}")
print(f"Features: {feature_names}")
print(f"Species: {list(species_names.values())}")
train_df.head()


"""**Cell 5 (Markdown):**"""
## Step 2: Initialize the Metadata Store
Using an in-memory database for this demo. In production, you'd use SQLite or MySQL.

In [ ]:
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.fake_database.SetInParent()
store = metadata_store.MetadataStore(connection_config)
print("ML Metadata store initialized!")



'''**Cell 7 (Markdown):**'''

## Step 3: Register Artifact Types
These define what kinds of artifacts our pipeline produces — datasets, schemas, models, and API configs.

In [ ]:
# Dataset artifact type
dataset_type = metadata_store_pb2.ArtifactType()
dataset_type.name = "IrisDataset"
dataset_type.properties["name"] = metadata_store_pb2.STRING
dataset_type.properties["split"] = metadata_store_pb2.STRING
dataset_type.properties["num_samples"] = metadata_store_pb2.INT
dataset_type.properties["num_features"] = metadata_store_pb2.INT
dataset_type.properties["version"] = metadata_store_pb2.INT
dataset_type_id = store.put_artifact_type(dataset_type)

# Schema artifact type (our custom schema instead of TFDV)
schema_type = metadata_store_pb2.ArtifactType()
schema_type.name = "DataSchema"
schema_type.properties["name"] = metadata_store_pb2.STRING
schema_type.properties["num_columns"] = metadata_store_pb2.INT
schema_type.properties["version"] = metadata_store_pb2.INT
schema_type_id = store.put_artifact_type(schema_type)

# Model artifact type (matching our FastAPI lab model)
model_type = metadata_store_pb2.ArtifactType()
model_type.name = "IrisModel"
model_type.properties["name"] = metadata_store_pb2.STRING
model_type.properties["algorithm"] = metadata_store_pb2.STRING
model_type.properties["accuracy"] = metadata_store_pb2.DOUBLE
model_type.properties["n_estimators"] = metadata_store_pb2.INT
model_type.properties["version"] = metadata_store_pb2.INT
model_type_id = store.put_artifact_type(model_type)

# API Config artifact type (linking to FastAPI lab)
api_type = metadata_store_pb2.ArtifactType()
api_type.name = "APIConfig"
api_type.properties["name"] = metadata_store_pb2.STRING
api_type.properties["framework"] = metadata_store_pb2.STRING
api_type.properties["endpoints"] = metadata_store_pb2.INT
api_type.properties["version"] = metadata_store_pb2.INT
api_type_id = store.put_artifact_type(api_type)

print(f"IrisDataset type ID: {dataset_type_id}")
print(f"DataSchema type ID: {schema_type_id}")
print(f"IrisModel type ID: {model_type_id}")
print(f"APIConfig type ID: {api_type_id}")



'''**Cell 9 (Markdown):**'''
## Step 4: Register Execution Types
These represent the steps in our pipeline — data validation, model training, and API deployment.

In [ ]:
# Data Validation step
validation_type = metadata_store_pb2.ExecutionType()
validation_type.name = "DataValidation"
validation_type.properties["state"] = metadata_store_pb2.STRING
validation_type_id = store.put_execution_type(validation_type)

# Model Training step
training_type = metadata_store_pb2.ExecutionType()
training_type.name = "ModelTraining"
training_type.properties["state"] = metadata_store_pb2.STRING
training_type.properties["algorithm"] = metadata_store_pb2.STRING
training_type_id = store.put_execution_type(training_type)

# API Deployment step
deploy_type = metadata_store_pb2.ExecutionType()
deploy_type.name = "APIDeployment"
deploy_type.properties["state"] = metadata_store_pb2.STRING
deploy_type.properties["framework"] = metadata_store_pb2.STRING
deploy_type_id = store.put_execution_type(deploy_type)

print(f"DataValidation type ID: {validation_type_id}")
print(f"ModelTraining type ID: {training_type_id}")
print(f"APIDeployment type ID: {deploy_type_id}")

'''**Cell 11 (Markdown):**'''
## Step 5: Run Data Validation (Custom Schema Generation)
Instead of TFDV, we generate our own schema using pandas — including stats that help catch data issues.

In [ ]:
# Create dataset artifact
train_artifact = metadata_store_pb2.Artifact()
train_artifact.uri = './pipeline_data/train/iris_train.csv'
train_artifact.type_id = dataset_type_id
train_artifact.properties["name"].string_value = "Iris Training Data"
train_artifact.properties["split"].string_value = "train"
train_artifact.properties["num_samples"].int_value = len(train_df)
train_artifact.properties["num_features"].int_value = 4
train_artifact.properties["version"].int_value = 1
train_artifact_id = store.put_artifacts([train_artifact])[0]

# Start validation execution
val_execution = metadata_store_pb2.Execution()
val_execution.type_id = validation_type_id
val_execution.properties["state"].string_value = "RUNNING"
val_execution_id = store.put_executions([val_execution])[0]

# Link dataset as input to validation
input_event = metadata_store_pb2.Event()
input_event.artifact_id = train_artifact_id
input_event.execution_id = val_execution_id
input_event.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([input_event])

print(f"Dataset artifact ID: {train_artifact_id}")
print(f"Validation execution ID: {val_execution_id} (RUNNING)")

In [ ]:
# Custom schema generation — our own version of what TFDV does
df = pd.read_csv('./pipeline_data/train/iris_train.csv')

schema = {"dataset": "Iris Training Data", "columns": {}}
for col in df.columns:
    col_schema = {
        "dtype": str(df[col].dtype),
        "nulls": int(df[col].isnull().sum()),
        "unique": int(df[col].nunique())
    }
    if df[col].dtype in ['float64', 'int64']:
        col_schema["min"] = round(float(df[col].min()), 4)
        col_schema["max"] = round(float(df[col].max()), 4)
        col_schema["mean"] = round(float(df[col].mean()), 4)
        col_schema["std"] = round(float(df[col].std()), 4)
    elif df[col].dtype == 'object':
        col_schema["categories"] = list(df[col].unique())
    schema["columns"][col] = col_schema

# Save schema
with open('./pipeline_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)

print("Custom Schema Generated!\n")
for col, info in schema["columns"].items():
    print(f"  {col}: {info['dtype']} | nulls={info['nulls']} | unique={info['unique']}")
    if 'mean' in info:
        print(f"    range=[{info['min']}, {info['max']}] mean={info['mean']}")

In [ ]:
# Register schema artifact and complete validation
schema_artifact = metadata_store_pb2.Artifact()
schema_artifact.uri = './pipeline_schema.json'
schema_artifact.type_id = schema_type_id
schema_artifact.properties["name"].string_value = "Iris Data Schema"
schema_artifact.properties["num_columns"].int_value = len(schema["columns"])
schema_artifact.properties["version"].int_value = 1
schema_artifact_id = store.put_artifacts([schema_artifact])[0]

# Output event
output_event = metadata_store_pb2.Event()
output_event.artifact_id = schema_artifact_id
output_event.execution_id = val_execution_id
output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([output_event])

# Mark validation as complete
val_execution.id = val_execution_id
val_execution.properties["state"].string_value = "COMPLETED"
store.put_executions([val_execution])

print(f"Schema artifact ID: {schema_artifact_id}")
print("Data Validation: COMPLETED")
```

**Cell 15 (Markdown):**
```
## Step 6: Train the Model (Same as FastAPI Lab)
Training a Random Forest just like in our FastAPI lab — with confidence scores.

In [ ]:
# Start training execution
train_execution = metadata_store_pb2.Execution()
train_execution.type_id = training_type_id
train_execution.properties["state"].string_value = "RUNNING"
train_execution.properties["algorithm"].string_value = "RandomForest"
train_execution_id = store.put_executions([train_execution])[0]

# Input: dataset -> training
train_input = metadata_store_pb2.Event()
train_input.artifact_id = train_artifact_id
train_input.execution_id = train_execution_id
train_input.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([train_input])

# Train model (same config as FastAPI lab)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Confidence scores (like our FastAPI lab)
probabilities = rf_model.predict_proba(X_test)
avg_confidence = np.mean(np.max(probabilities, axis=1))

os.makedirs('pipeline_model', exist_ok=True)
joblib.dump(rf_model, 'pipeline_model/iris_rf.pkl')

print(f"Model Accuracy: {acc*100:.2f}%")
print(f"Average Confidence: {avg_confidence*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=list(species_names.values())))

In [ ]:
# Register model artifact
model_artifact = metadata_store_pb2.Artifact()
model_artifact.uri = './pipeline_model/iris_rf.pkl'
model_artifact.type_id = model_type_id
model_artifact.properties["name"].string_value = "Iris Random Forest v1"
model_artifact.properties["algorithm"].string_value = "RandomForest"
model_artifact.properties["accuracy"].double_value = acc
model_artifact.properties["n_estimators"].int_value = 100
model_artifact.properties["version"].int_value = 1
model_artifact_id = store.put_artifacts([model_artifact])[0]

# Output event
model_output = metadata_store_pb2.Event()
model_output.artifact_id = model_artifact_id
model_output.execution_id = train_execution_id
model_output.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([model_output])

# Complete training
train_execution.id = train_execution_id
train_execution.properties["state"].string_value = "COMPLETED"
store.put_executions([train_execution])

print(f"Model artifact ID: {model_artifact_id}")
print("Model Training: COMPLETED")
```

**Cell 18 (Markdown):**
```
## Step 7: Record API Deployment (Linking to FastAPI Lab)
This tracks the deployment of our model as a FastAPI service with confidence scores and species descriptions.

In [ ]:
# Start deployment execution
deploy_execution = metadata_store_pb2.Execution()
deploy_execution.type_id = deploy_type_id
deploy_execution.properties["state"].string_value = "RUNNING"
deploy_execution.properties["framework"].string_value = "FastAPI"
deploy_execution_id = store.put_executions([deploy_execution])[0]

# Input: model -> deployment
deploy_input = metadata_store_pb2.Event()
deploy_input.artifact_id = model_artifact_id
deploy_input.execution_id = deploy_execution_id
deploy_input.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([deploy_input])

# Create API config artifact
api_config = {
    "title": "Iris Flower Prediction API",
    "version": "2.0.0",
    "model": "RandomForest",
    "endpoints": [
        {"method": "GET", "path": "/", "description": "Health check"},
        {"method": "POST", "path": "/predict", "description": "Predict with confidence"},
        {"method": "GET", "path": "/model-info", "description": "Model details"},
        {"method": "GET", "path": "/species", "description": "Species info"}
    ],
    "features": ["confidence_scores", "species_descriptions", "input_validation"]
}

with open('./api_config.json', 'w') as f:
    json.dump(api_config, f, indent=2)

api_artifact = metadata_store_pb2.Artifact()
api_artifact.uri = './api_config.json'
api_artifact.type_id = api_type_id
api_artifact.properties["name"].string_value = "Iris Prediction API v2.0"
api_artifact.properties["framework"].string_value = "FastAPI"
api_artifact.properties["endpoints"].int_value = 4
api_artifact.properties["version"].int_value = 1
api_artifact_id = store.put_artifacts([api_artifact])[0]

# Output event
api_output = metadata_store_pb2.Event()
api_output.artifact_id = api_artifact_id
api_output.execution_id = deploy_execution_id
api_output.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([api_output])

# Complete deployment
deploy_execution.id = deploy_execution_id
deploy_execution.properties["state"].string_value = "COMPLETED"
store.put_executions([deploy_execution])

print(f"API artifact ID: {api_artifact_id}")
print("API Deployment: COMPLETED")
print(f"Endpoints tracked: {len(api_config['endpoints'])}")
```

**Cell 20 (Markdown):**
```
## Step 8: Set Up Context (Group the Entire Pipeline)

In [ ]:
# Create context type
pipeline_type = metadata_store_pb2.ContextType()
pipeline_type.name = "MLPipeline"
pipeline_type.properties["description"] = metadata_store_pb2.STRING
pipeline_type.properties["owner"] = metadata_store_pb2.STRING
pipeline_type_id = store.put_context_type(pipeline_type)

# Create context
pipeline_context = metadata_store_pb2.Context()
pipeline_context.type_id = pipeline_type_id
pipeline_context.name = "IrisFlowerPipeline_v1"
pipeline_context.properties["description"].string_value = "End-to-end Iris prediction pipeline: data -> schema -> model -> API"
pipeline_context.properties["owner"].string_value = "Hrishikesh Prabhu"
context_id = store.put_contexts([pipeline_context])[0]

# Link all artifacts
all_artifact_ids = [train_artifact_id, schema_artifact_id, model_artifact_id, api_artifact_id]
for art_id in all_artifact_ids:
    attr = metadata_store_pb2.Attribution()
    attr.artifact_id = art_id
    attr.context_id = context_id
    store.put_attributions_and_associations([attr], [])

# Link all executions
all_execution_ids = [val_execution_id, train_execution_id, deploy_execution_id]
for exec_id in all_execution_ids:
    assoc = metadata_store_pb2.Association()
    assoc.execution_id = exec_id
    assoc.context_id = context_id
    store.put_attributions_and_associations([], [assoc])

print(f"Pipeline context ID: {context_id}")
print(f"Linked {len(all_artifact_ids)} artifacts and {len(all_execution_ids)} executions")
```

**Cell 22 (Markdown):**
```
## Step 9: Query the Metadata Store — Full Pipeline Lineage
Let's trace the entire pipeline and answer key questions.

In [ ]:
print("=" * 60)
print("FULL PIPELINE SUMMARY")
print("=" * 60)

# All artifacts
print("\n--- Artifacts ---")
for art_type in store.get_artifact_types():
    for a in store.get_artifacts_by_type(art_type.name):
        print(f"  [{art_type.name}] {a.uri}")

# All executions
print("\n--- Executions ---")
for exec_type in store.get_execution_types():
    for e in store.get_executions_by_type(exec_type.name):
        state = e.properties["state"].string_value
        print(f"  [{exec_type.name}] State: {state}")

# Lineage: What dataset produced the API?
print("\n--- Lineage: API -> Model -> Dataset ---")
api_events = store.get_events_by_artifact_ids([api_artifact_id])
for event in api_events:
    if event.type == metadata_store_pb2.Event.DECLARED_OUTPUT:
        deploy_events = store.get_events_by_execution_ids([event.execution_id])
        for de in deploy_events:
            if de.type == metadata_store_pb2.Event.DECLARED_INPUT:
                model = store.get_artifacts_by_id([de.artifact_id])[0]
                print(f"  API served by: {model.properties['name'].string_value}")
                print(f"  Model accuracy: {model.properties['accuracy'].double_value*100:.2f}%")
                
                # Go deeper — what dataset trained this model?
                model_events = store.get_events_by_artifact_ids([model.id])
                for me in model_events:
                    if me.type == metadata_store_pb2.Event.DECLARED_OUTPUT:
                        train_events = store.get_events_by_execution_ids([me.execution_id])
                        for te in train_events:
                            if te.type == metadata_store_pb2.Event.DECLARED_INPUT:
                                dataset = store.get_artifacts_by_id([te.artifact_id])[0]
                                print(f"  Trained on: {dataset.properties['name'].string_value}")
                                print(f"  Dataset: {dataset.uri}")
                                print(f"  Samples: {dataset.properties['num_samples'].int_value}")

In [ ]:
 # Context query — get everything in our pipeline
print("=" * 60)
print("EVERYTHING IN OUR PIPELINE CONTEXT")
print("=" * 60)

context_artifacts = store.get_artifacts_by_context(context_id)
context_executions = store.get_executions_by_context(context_id)

print(f"\nArtifacts ({len(context_artifacts)}):")
for a in context_artifacts:
    print(f"  - {a.uri}")

print(f"\nExecutions ({len(context_executions)}):")
for e in context_executions:
    print(f"  - State: {e.properties['state'].string_value}")
```

**Cell 25 (Markdown):**
```
## Summary — How All My Labs Connect

| Lab | What I Did | Connection |
|---|---|---|
| **GitHub Lab 1** | Built calculator toolkit with pytest + GitHub Actions CI/CD | Learned testing fundamentals |
| **FastAPI Lab** | Served Iris model as API with confidence scores & species info | Applied testing to API endpoints |
| **MLMD Lab** | Tracked the entire Iris pipeline with ML Metadata | Recorded metadata for the same model served in FastAPI |

The key insight: In production, you don't just train a model and deploy it. You need to track *which dataset* was used, *what schema* it followed, *which model version* is deployed, and *how the API is configured*. ML Metadata gives you that full lineage — so if something breaks in production, you can trace back to exactly what changed.